In [ ]:
import os
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

from utils import save_object
from logger import logging
from exception import CustomException

ModuleNotFoundError: No module named 'utils'

In [ ]:
DATA_CSV   = "data/isl_keypoints.csv"
MODEL_DIR  = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "isl_model.pkl")
ENC_PATH   = os.path.join(MODEL_DIR, "label_encoder.pkl")
SCALER_PATH = os.path.join(MODEL_DIR, "scaler.pkl")

In [ ]:
def train():
    try:
        logging.info("Loading dataset...")
        df = pd.read_csv(DATA_CSV)
        print(f"Dataset: {df.shape[0]} rows, {df['label'].nunique()} classes")
        print(f"Classes: {sorted(df['label'].unique())}")

        X = df.drop("label", axis=1).values.astype(np.float32)
        y = df["label"].values

        #Encoding A-Z to 0-25
        le = LabelEncoder()
        y_enc = le.fit_transform(y)

        #Scaling features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
        )
        print(f"Train: {len(X_train)} | Test: {len(X_test)}")

        #MLP Neural Network
        print("\nTraining ANN... (this takes 1-3 mins)")
        model = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=42,
            verbose=True
        )

        model.fit(X_train, y_train)

        #Evaluate
        y_pred = model.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)

        print(f"\n Test Accuracy: {acc*100:.2f}%")
        print(classification_report(y_test, y_pred,
                                    target_names=le.classes_))
        logging.info(f"Test accuracy: {acc*100:.2f}%")

        #Save everything
        os.makedirs(MODEL_DIR, exist_ok=True)
        save_object(MODEL_PATH,  model)
        save_object(ENC_PATH,    le)
        save_object(SCALER_PATH, scaler)

        print(f"\nModel saved   → {MODEL_PATH}")
        print(f"Encoder saved → {ENC_PATH}")
        print(f"Scaler saved  → {SCALER_PATH}")

    except Exception as e:
        raise CustomException(e, sys)

if __name__ == "__main__":
    train()

NameError: name 'CustomException' is not defined